# Text-level Tests

**Prerequisites**: run all cells in `main.ipynb` first so `Cathey`, `memory` objects are available,
or re-initialize everything in Cell 1 below.


In [ ]:
# Cell 1: Bootstrap (skip if already initialized from main.ipynb)
import sys, os

# Add project root so local modules are importable from this subdirectory
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("text_test.ipynb"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

from sentence_transformers import SentenceTransformer
from nlp.llm_parser import LLMParser
from core.memory import MemoryManager
from core.agent import CatheyAgent
from hardware.audio import TTSEngine

_embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed = lambda t: _embed_model.encode(t, convert_to_numpy=True).tolist()

llm    = LLMParser()
tts    = TTSEngine()
memory = MemoryManager(embed_fn=embed)
Cathey   = CatheyAgent(llm=llm, memory=memory, speak=tts.speak)
print("Cathey ready.")

## Group A — Basic functionality regression
Each test is isolated (state reset before each).

In [ ]:
group_a = [
    ("Hello Cathey, turn on the light.",   "direct_command"),
    ("Cathey, turn off the light.",        "direct_command"),
    ("Cathey, set the AC to 24 degrees.",  "direct_command"),
    ("Cathey, I feel a little bit cold.",  "needs_clarification"),
    ("Cathey, it's a bit dark.",           "needs_clarification"),
    ("Cathey, fuck this light.",           "needs_clarification"),
    ("Cathey, make this room lively.",     "needs_clarification"),
    ("Cathey, how can I eat an apple?",    "general_qa"),
    ("Cathey, can I still eat this dish after one night in the fridge?", "general_qa"),
    ("How are you today?",               "invalid/prefilter"),
]

print(f"{'Input':<55} {'Expected':<22} {'Got':<22} {'Pass?'}")
print("-" * 110)

passed = 0
for text, expected_label in group_a:
    Cathey.reset_dialogue()
    result = Cathey.handle(text, verbose=False)
    got = result["semantic"].get("type", "N/A")
    pre = result["prefilter_passed"]
    # For prefilter cases, check prefilter_passed is False
    if "prefilter" in expected_label:
        ok = not pre
    else:
        ok = got == expected_label
    passed += ok
    mark = "✓" if ok else "✗"
    print(f"{text:<55} {expected_label:<22} {got:<22} {mark}")

print(f"\nGroup A: {passed}/{len(group_a)} passed")

## Group B — Memory system demonstration
Stateful, continuous — verifies procedural memory learning.

In [ ]:
# Reset memory to start fresh
memory.clear_working()

# B1: First time "I feel cold" → clarification
print("[B1] First time: Cathey, I feel cold — expect clarification")
Cathey.reset_dialogue()
r = Cathey.handle("Cathey, I feel cold.", verbose=False)
print(f"  execution={r.get('execution')}")
print(f"  reply: {r.get('spoken_reply','')[:80]}")

# B1 followup: user picks option 1
print("\n[B1 followup] User: option 1")
r = Cathey.handle("option 1", verbose=False)
print(f"  execution={r.get('execution')}")
print(f"  skills learned: {len(memory.skills)}")

# B2: Same intent again → procedural memory auto-resolves without asking
print("\n[B2] Second time: Cathey, I feel a little cold — expect AUTO-RESOLVED")
Cathey.reset_dialogue()
r = Cathey.handle("Cathey, I feel a little cold.", verbose=False)
print(f"  reason={r.get('reason')}")
print(f"  reply: {r.get('spoken_reply','')[:80]}")

# B3: General QA with working memory context
print("\n[B3] General QA (RAG should see B1 history)")
Cathey.reset_dialogue()
r = Cathey.handle("Cathey, can I still eat this dish after a night in the fridge?", verbose=False)
print(f"  answer: {r.get('spoken_reply','')[:120]}")

# B4: Memory summary
print("\n[B4] Memory state:")
for k, v in memory.summary().items():
    print(f"  {k}: {v}")

## Batch Evaluation

In [ ]:
import time, json, pandas as pd

batch_cases = [
    {"input": "Cathey, I feel a little bit cold.",                       "expected_type": "needs_clarification"},
    {"input": "Cathey, it's a bit dark.",                                "expected_type": "needs_clarification"},
    {"input": "Cathey, fuck this light.",                                "expected_type": "needs_clarification"},
    {"input": "Cathey, make this room lively.",                          "expected_type": "needs_clarification"},
    {"input": "Cathey, how can I eat an apple?",                         "expected_type": "general_qa"},
    {"input": "Cathey, can I still eat this dish after one night?",      "expected_type": "general_qa"},
]

rows = []
for item in batch_cases:
    Cathey.reset_dialogue()
    t0 = time.perf_counter()
    result = Cathey.handle(item["input"], verbose=False)
    ms = (time.perf_counter() - t0) * 1000
    semantic = result.get("semantic", {})
    got_type = semantic.get("type", "N/A")
    match = got_type == item["expected_type"]
    rows.append({
        "input":        item["input"],
        "expected":     item["expected_type"],
        "got":          got_type,
        "match":        match,
        "latency_ms":   round(ms, 1),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nAccuracy: {df['match'].mean():.0%}  |  Avg latency: {df['latency_ms'].mean():.0f} ms")